# Scheduling observations by expected information gain

This notebook builds photomancy's value-of-information loop from the ground up: it opens with
the theory of how an observation is scored, then puts that score in a closed loop that decides
what to observe next. The demonstration is a radial-velocity period search, the setting where
information-driven scheduling has the most to gain over a fixed cadence, but the loop itself is
generic, the same `Posterior`, `evaluate_candidates`, and `to_prior` that drive scheduling at
every scale.


## The value of an observation

An observation is worth taking to the extent that it is expected to teach us something.
photomancy makes that precise with the expected information gain, the mutual information between
the parameters $z$ and a candidate observation $Y$,

$$ \mathrm{EIG}(Y) = I(z; Y) = \mathbb{E}_{Y}\big[\, D_{\mathrm{KL}}\!\big(p(z \mid Y) \,\|\, p(z)\big) \,\big], $$

equivalently the expected reduction in the entropy of the belief once the data arrive. A
scheduler that maximizes this quantity points where the next measurement is expected to sharpen
the posterior the most. The [mathematical foundations](../explanation/mathematical-foundations)
derive the terms used below.


## Two ways an observation informs

For a belief held as a Gaussian mixture, the gain splits into two readable pieces. Within a
single mode, linearizing the forward model about the mode, $y = J z + \varepsilon$ with Jacobian
$J = \partial y / \partial z$ and measurement covariance $R$, the matrix $J^\top R^{-1} J$ is
the Fisher information the observation carries about $z$ (the variable named `fim` in
`photomancy.eig`). The posterior precision updates by adding it to the prior precision,
$\Sigma_{\mathrm{new}}^{-1} = \Sigma_{\mathrm{old}}^{-1} + J^\top R^{-1} J$, and the information
gained is the log-volume contraction of the covariance,

$$ \mathrm{EIG}_{\mathrm{geom}} = \tfrac{1}{2}\big(\log|\Sigma_{\mathrm{old}}| - \log|\Sigma_{\mathrm{new}}|\big). $$

Maximizing this term over candidates is Bayesian D-optimal design. Across modes, an
observation helps instead by telling them apart. With per-mode predictions $y_k$,
weights $w_k$, and weighted variance $\mathrm{Var}(y)$ per observable, the alias-breaking gain is

$$ \mathrm{EIG}_{\mathrm{alias}} = \tfrac{1}{2}\sum_{\text{obs}} \log\!\Big(1 + \frac{\mathrm{Var}(y)}{R}\Big). $$

The first term is the Fisher-information gain that refines a well-isolated estimate, the second
resolves which of several competing explanations is correct and is the part the Fisher
information cannot describe, since it is a property of the discrete spread across modes.
`photomancy.eig.evaluate_candidates` evaluates both over a batch of candidates against a
mixture posterior, returning the total and each part.


## The scheduling loop

A myopic scheduler chains the gain into a loop. Given the current belief, it scores the
candidate observations, takes the most informative one, folds the result into the belief, and
repeats:

1. score every candidate with the expected information gain,
2. observe at the candidate that maximizes it,
3. update the belief, where the current posterior becomes the next prior through `to_prior`,
4. repeat.

The problem below is built so that the two gain terms matter in turn. A radial-velocity signal
is sampled on a nightly cadence, which leaves the period aliased, so the belief starts multimodal
and the early gain is alias-breaking. Once the alias is resolved, the belief is a single mode and
the gain becomes geometric, refining the period.


In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

from photomancy import LaplaceMixtureBackend, build_scene_logdensity
from photomancy.eig import evaluate_candidates
from photomancy.priors import Uniform

PI = jnp.pi


class RV(eqx.Module):
    # the scene: a radial-velocity signal's period (as log P), amplitude, and phase
    logP: jnp.ndarray
    K: jnp.ndarray
    phi: jnp.ndarray


def rv(p, t):
    return p.K * jnp.sin(2 * PI * t / 10.0**p.logP + p.phi)


P_true = 2.7
f_true = 1.0 / P_true
# integer-day sampling aliases a period with those at f_true + n cycles/day
alias_P = np.array([P_true, 1.0 / (f_true + 1.0), 1.0 / (f_true + 2.0)])
truth = RV(logP=jnp.array(np.log10(P_true)), K=jnp.array(5.0), phi=jnp.array(0.7))
err = 0.4  # m/s

t_init = jnp.arange(0.0, 8.0)  # eight nightly epochs
y_init = rv(truth, t_init) + err * jax.random.normal(jax.random.key(0), t_init.shape)

prior = Uniform(
    low=jnp.array([np.log10(0.3), 0.0, -PI]),
    high=jnp.array([np.log10(5.0), 10.0, PI]),
)
inits = jnp.array([[np.log10(P), 4.0, 0.5] for P in alias_P])  # one start per alias


def fit(t_obs, y_obs, starts):
    def forward(p):
        return rv(p, t_obs)

    def likelihood(pred):
        return -0.5 * jnp.sum(((pred - y_obs) / err) ** 2)

    ld, _, _ = build_scene_logdensity(
        RV(logP=jnp.array(0.0), K=jnp.array(0.0), phi=jnp.array(0.0)),
        forward, likelihood, prior,
    )
    return LaplaceMixtureBackend(n_steps=400, min_eigenvalue=1e-8).run(ld, starts)


def eig_forward(z, t):
    return jnp.atleast_1d(z[1] * jnp.sin(2 * PI * t / 10.0**z[0] + z[2]))


def periods_weights(post):
    return 10.0 ** np.array(post.means[:, 0]), np.array(jnp.exp(post.log_weights))


def true_weight(post):
    P, w = periods_weights(post)
    return float(w[np.argmin(np.abs(P - P_true))])


def observe(t_obs, y_obs, t_star, key):
    y_star = rv(truth, jnp.asarray(t_star)) + err * jax.random.normal(key, ())
    return (
        jnp.concatenate([t_obs, jnp.atleast_1d(jnp.asarray(t_star))]),
        jnp.concatenate([y_obs, jnp.atleast_1d(y_star)]),
    )


## A problem with aliases

The signal is a single sinusoid, a planet's radial-velocity wobble
$v(t) = K \sin(2\pi t / P + \varphi)$, observed on eight consecutive nights. Because the samples
fall on integer days, the true period and the shorter periods whose frequencies differ from
$1/P$ by a whole number of cycles per day reproduce the same eight values, so the data alone
cannot tell them apart.


In [ ]:
tt = np.linspace(-0.5, 8.0, 600)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(tt, np.array(rv(truth, jnp.array(tt))), color="0.7", lw=1, label="true signal")
ax.scatter(np.array(t_init), np.array(y_init), c="crimson", s=35, zorder=3, label="nightly data")
ax.set_xlabel("time [days]")
ax.set_ylabel("radial velocity [m/s]")
ax.set_title("eight nightly radial-velocity measurements")
ax.legend()
plt.show()


## The multimodal belief

A multi-start Laplace fit, one start per candidate period, returns an evidence-weighted mixture.
The three modes are the true period and its two nightly aliases, and they carry comparable weight
because they fit the nightly data almost equally. Plotted as curves they are nearly
indistinguishable at the observed nights and disagree everywhere between them, which is exactly
the structure an observation can exploit.


In [ ]:
post0 = fit(t_init, y_init, inits)
P, w = periods_weights(post0)
for i in np.argsort(-w):
    print(f"mode: P = {P[i]:.3f} d   weight = {w[i]:.3f}")

tt = np.linspace(0.0, 8.0, 800)
fig, ax = plt.subplots(figsize=(7, 3.2))
for i in np.argsort(-w):
    m = post0.means[i]
    curve = float(m[1]) * np.sin(2 * np.pi * tt / 10.0 ** float(m[0]) + float(m[2]))
    ax.plot(tt, curve, lw=1.3, label=f"P = {P[i]:.2f} d  (w = {w[i]:.2f})")
ax.scatter(np.array(t_init), np.array(y_init), c="crimson", s=35, zorder=3)
ax.set_xlabel("time [days]")
ax.set_ylabel("radial velocity [m/s]")
ax.set_title("three period modes: agree at the nights, disagree between")
ax.legend(fontsize=8)
plt.show()


## Scoring candidate observations

The expected information gain over candidate observation times is dominated by the alias-breaking
term, and it peaks between the nights, where the three periods predict the most different
velocities. At the integer days it nearly vanishes, since there the aliases agree by
construction. The scheduler therefore wants an off-cadence epoch, the opposite of simply
continuing the nightly run.


In [ ]:
cand = jnp.linspace(0.0, 24.0, 480)
e = evaluate_candidates(post0, cand, eig_forward, err**2)
i_best = int(jnp.argmax(e["total_eig"]))
t_star = float(cand[i_best])

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(np.array(cand), np.array(e["total_eig"]), color="navy", label="total")
ax.plot(np.array(cand), np.array(e["alias_eig"]), color="darkorange", lw=1, label="alias-breaking")
ax.plot(np.array(cand), np.array(e["geometric_eig"]), color="seagreen", lw=1, label="geometric")
for d in range(0, 25):
    ax.axvline(d, color="0.92", lw=0.6, zorder=0)
ax.axvline(t_star, color="crimson", ls="--", lw=1.2, label=f"best t* = {t_star:.1f} d")
ax.set_xlabel("candidate observation time [days]")
ax.set_ylabel("EIG [nats]")
ax.set_title("expected information gain over candidate times")
ax.legend(fontsize=8)
plt.show()
print(f"best next time t* = {t_star:.2f} d  ({float(e['total_eig'][i_best]):.2f} nats)")


## One observation breaks the alias

Taking the single most informative epoch and folding it into the belief collapses the mixture.
The true period keeps essentially all of the weight, and the aliases, which predicted the wrong
velocity at that off-cadence time, are ruled out.


In [ ]:
t1, y1 = observe(t_init, y_init, t_star, jax.random.key(7))
post1 = fit(t1, y1, post0.means)
P1, w1 = periods_weights(post1)

x = np.arange(len(alias_P))
w0_by = np.array([w[np.argmin(np.abs(P - p))] for p in alias_P])
w1_by = np.array([w1[np.argmin(np.abs(P1 - p))] for p in alias_P])
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(x - 0.18, w0_by, width=0.36, label="before", color="0.7")
ax.bar(x + 0.18, w1_by, width=0.36, label="after one epoch", color="crimson")
ax.set_xticks(x)
ax.set_xticklabels([f"{p:.2f} d" for p in alias_P])
ax.set_xlabel("period mode")
ax.set_ylabel("posterior weight")
ax.set_title(f"one observation at t* = {t_star:.1f} d collapses the alias")
ax.legend()
plt.show()


## The full loop versus a fixed cadence

Run end to end, the information-driven schedule resolves the alias on its first extra epoch and
then extends the baseline to sharpen the period. A fixed nightly cadence, the natural default,
never resolves the alias at all, because every integer-day sample leaves the periods degenerate,
so the belief keeps a stubborn three-way ambiguity no matter how many nights are added. For the
same number of observations the information-driven loop reaches a far tighter period.


In [ ]:
def run_loop(strategy, n_steps=6):
    t_obs, y_obs = t_init, y_init
    post = fit(t_obs, y_obs, inits)
    starts = post.means
    rec = []
    for step in range(n_steps):
        if strategy == "greedy":
            t_last = float(jnp.max(t_obs))
            c = jnp.linspace(t_last + 0.3, t_last + 10.0, 200)  # a forward horizon
            ev = evaluate_candidates(post, c, eig_forward, err**2)
            t_star = c[int(jnp.argmax(ev["total_eig"]))]
        else:
            t_star = jnp.array(8.0 + step * 3.0)  # keep adding nights
        t_obs, y_obs = observe(t_obs, y_obs, t_star, jax.random.key(100 + step))
        post = fit(t_obs, y_obs, starts)
        starts = post.means
        best = int(np.argmax(np.array(jnp.exp(post.log_weights))))
        rec.append((true_weight(post), float(jnp.sqrt(post.covs[best, 0, 0]))))
    return np.array(rec)


g, u = run_loop("greedy"), run_loop("uniform")
steps = np.arange(1, len(g) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.4))
a1.plot(steps, g[:, 0], "o-", color="navy", label="information-driven")
a1.plot(steps, u[:, 0], "s--", color="0.55", label="fixed cadence")
a1.set_xlabel("extra observations")
a1.set_ylabel("weight on the true period")
a1.set_ylim(0, 1.05)
a1.legend(fontsize=8)
a2.plot(steps, g[:, 1], "o-", color="navy", label="information-driven")
a2.plot(steps, u[:, 1], "s--", color="0.55", label="fixed cadence")
a2.set_xlabel("extra observations")
a2.set_ylabel("period uncertainty, sigma(log P)")
a2.set_yscale("log")
a2.legend(fontsize=8)
fig.suptitle("information-driven scheduling resolves the alias and sharpens the period")
fig.tight_layout()
plt.show()


## What this is

This loop is the photomancy scheduler in miniature. The belief is a `Posterior`, the candidate
score is `evaluate_candidates`, and the update is `to_prior` followed by a refit, the same three
pieces that drive scheduling at every scale. Swapping the radial-velocity forward model for an
orbit, an atmosphere, or a coronagraph image reuses the loop unchanged, which is the point of the
[architecture](../explanation/architecture): one information currency, computed the same way,
from a single exposure up to a whole mission. The orbit layer ships its own
`evaluate_orbit_candidates` for exactly this, and the detectability term extends the same idea to
the question of whether a planet is there to be seen at all. The
[abstract walkthrough](walkthrough) shows the same machinery on a simpler spatial problem.
